In [1]:
from pathlib import Path
import json
from part3_storage import storage
stage2=json.loads(Path('part2_collection/status.json').read_text(encoding='utf-8'))
result=json.loads(Path('part3_storage/evidence/real_pipeline_result.json').read_text(encoding='utf-8'))
print('实验二：按用户要求保存前8页；浏览器真实DOM + Python解析')
print('原始条数:',stage2['raw_count'],' 完全重复:',stage2['raw_count']-stage2['deduplicated_count'],' CSV条数:',stage2['deduplicated_count'])
print('首次入库日志：前',result['before_count'],' 后',result['after_count'],' 新增',result['inserted_count'])
print('新进程独立重跑新增:',result['rerun_inserted_count'])
connection=cursor=None
try:
    connection=storage.connect(); cursor=connection.cursor()
    cursor.execute('SELECT COUNT(*) FROM houseinfo')
    live_count=cursor.fetchone()[0]
    print('当前真实查询：SELECT COUNT(*) FROM houseinfo =',live_count)
    assert live_count==437 and result['state']=='ready' and result['rerun_inserted_count']==0
    for item in result['three_way_samples']:
        if item['phase']!='run': continue
        cursor.execute('SELECT '+storage.COLS+' FROM houseinfo WHERE id=%s',(item['database_ids'][0],))
        actual=cursor.fetchone()
        expected=tuple(item['record'][f] for f in storage.FIELDS)
        assert actual==expected
        print('\n样本:',item['sample'],'网页第',item['page'],'页第',item['page_row'],'条；CSV第',item['csv_line'],'行；8字段一致')
        for field,value in zip(storage.FIELDS,actual): print('  '+field+': '+value)
    print('\nREAL_FIRST8_VERIFIED：CSV/数据库/三类网页来源样本核对通过')
finally:
    if connection is not None: connection.rollback()
    if cursor is not None: cursor.close()
    if connection is not None: connection.close()


实验二：按用户要求保存前8页；浏览器真实DOM + Python解析
原始条数: 480  完全重复: 43  CSV条数: 437
首次入库日志：前 0  后 437  新增 437
新进程独立重跑新增: 0
当前真实查询：SELECT COUNT(*) FROM houseinfo = 437

样本: first 网页第 1 页第 1 条；CSV第 2 行；8字段一致
  title: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
  house: 万科金域学府翰林菁华里
  address: 南岸学府大道二塘路59号
  struct: 3室2厅1卫
  area: 89.9㎡
  floor: 低层(共27层)
  maketime: 2019年建造
  price: 16465元/㎡

样本: cross_page 网页第 2 页第 1 条；CSV第 62 行；8字段一致
  title: 万科物业自营 朝中庭 业主诚心出售
  house: 万科金域学府翰林菁华里
  address: 南岸学府大道二塘路59号
  struct: 3室2厅1卫
  area: 85.7㎡
  floor: 低层(共27层)
  maketime: 2018年建造
  price: 16914元/㎡

样本: last 网页第 8 页第 60 条；CSV第 438 行；8字段一致
  title: 中央公园高品质洋房 出门就是轻轨站  三轻轨线环绕 总高6楼
  house: 荣盛鹿山府
  address: 渝北中央公园金州大道,近万康路
  struct: 3室2厅2卫
  area: 99㎡
  floor: 中层(共6层)
  maketime: 2020年建造
  price: 15960元/㎡

REAL_FIRST8_VERIFIED：CSV/数据库/三类网页来源样本核对通过
